# 02 — Inside the embedding

What the Schmidt decomposition actually does, why N₂ correctly gets *no*
bath, and how to read the diagnostics.

Uses the validated reference pickles in `tests/regression/golden/`, so it
runs in seconds and needs no PySCF.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

GOLDEN = Path("../tests/regression/golden")

def load(system, stage):
    with open(GOLDEN / system / f"{stage}.pkl", "rb") as fh:
        return pickle.load(fh)

systems = ["LiH", "N2", "ScH"]
step2 = {s: load(s, "step2_hamiltonian") for s in systems}

## The Schmidt spectrum decides the bath

DMET splits the molecule into an impurity (the active space) and its
environment. The singular values of the impurity–environment block of the
reference density measure how entangled the two are. Large values mean
environment orbitals worth pulling into the calculation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, s in zip(axes, systems):
    sv = np.asarray(step2[s]["sv_all"])
    n_bath = step2[s]["n_bath"]
    colours = ["#C44E52" if i < n_bath else "#4C72B0" for i in range(len(sv))]
    ax.bar(range(len(sv)), sv, color=colours)
    ax.set_title(f"{s}: n_bath = {n_bath}")
    ax.set_xlabel("singular value index")
    print(f"{s:4}  max|sv| = {np.max(np.abs(sv)):.3e}   n_bath = {n_bath}")

axes[0].set_ylabel("Schmidt singular value")
fig.tight_layout()

## N₂ has no bath, and that is correct

Every singular value is around 1e-15 — numerically zero. The active
orbitals are already close to eigenvectors of the reference density, so
there is no impurity–environment entanglement left to extract.

The tolerance is `1e-8`. Nothing clears it, so the bath is empty and the
embedding is the active space alone.

Taking the largest values anyway — "there must be *some* bath" — builds
physics out of rounding noise. On N₂ that produced a badly non-orthonormal
embedding basis and roughly **20 Ha** of error.

In [ ]:
from quenais.embedding.dmet_lib import adaptive_bath

sv_n2 = np.asarray(step2["N2"]["sv_all"])
print("N2 singular values:", sv_n2)
print()

n_bath, gap, cov = adaptive_bath(sv_n2, n_imp=4, max_embed=18, bath_tol=1e-8)
print(f"adaptive_bath -> n_bath={n_bath}, gap={gap}, coverage={cov}")

# What the pre-fix fallback would have done:
print(f"the old fallback would have taken the top {min(4, len(sv_n2))} anyway")

## The electron count comes from the density, not the active space

LiH's active space holds 2 electrons. Its *embedding* space — impurity
plus two bath orbitals — holds 4.

Deriving the count from the active space gives (1α, 1β). The reference
density says (2, 2). The wrong count roughly doubles the energy, and
nothing crashes.

In [ ]:
for s in systems:
    d2 = step2[s]
    a = float(np.sum(d2["ref_occ_alpha"]))
    b = float(np.sum(d2["ref_occ_beta"]))
    print(f"{s:4}  n_bath={d2['n_bath']}  "
          f"ref_occ sums = ({a:.6f}, {b:.6f})  "
          f"-> ({d2['n_alpha']}, {d2['n_beta']})")

## The independent check

`embedded_scf_check` compares a real SCF on the embedding Hamiltonian
against the full-molecule UHF energy. If the embedding Hamiltonian is
wrong, this fails — regardless of μ or the reference-density method.

In [ ]:
for s in systems:
    check = step2[s].get("embedded_scf_check")
    if check:
        print(f"{s:4}  delta = {check['delta']:+.3e} Ha   "
              f"within tolerance: {check['within_tol']}")
    else:
        print(f"{s:4}  (pickle predates the check)")

## Comparing your own run against these

`tools/compare_pickles.py` diffs any stage output against a golden one,
key by key, with per-quantity tolerances. It is what catches the failure
mode this project kept hitting: right shape, plausible magnitude, wrong
value.

```bash
python tools/compare_pickles.py \
    tests/regression/golden/LiH/step2_hamiltonian.pkl \
    ./lih_run/results/step2_hamiltonian.pkl -v
```

In [ ]:
import sys
sys.path.insert(0, "../tools")
from compare_pickles import compare

# Same file against itself: everything passes, nothing skipped.
report = compare(step2["LiH"], step2["LiH"])
print(report.summary())

# Now inject the electron-count bug and see it caught.
import copy
broken = copy.deepcopy(step2["LiH"])
broken["n_alpha"] = 1
print()
print(compare(step2["LiH"], broken).render())